# Phase 19: Reproducible heterogeneous BO benchmark

This notebook compares **Random**, **scrambled Sobol**, and **qEI** on the synthetic Strength objective. It uses paired initial designs across seeds and equal additional function-evaluation budgets. It is a **single-objective regression baseline**, not a demonstration of classification-constrained heterogeneous optimization.

In [ ]:
import matplotlib.pyplot as plt
import torch

from robotorchan.benchmarks.heterogeneous_multiseed import (
    compare_strategies,
    summarize_trajectories,
)

SEEDS = (1901, 1902, 1903)
INITIAL_POINTS = 12
STEPS = 3

## Execute paired trials

All strategies within a seed start from identical initial points and noisy initial observations. qEI refits its GP after every new observation. Increase `SEEDS` for meaningful experimental estimates; the defaults keep the notebook reasonably small.

In [ ]:
curves = compare_strategies(SEEDS, initial_points=INITIAL_POINTS, steps=STEPS)
assert set(curves) == {"random", "sobol", "qei"}
for name, values in curves.items():
    assert values.shape == (len(SEEDS), STEPS + 1)
    assert torch.isfinite(values).all()
    assert (values[:, 1:] >= values[:, :-1]).all()
    print(name, "per-seed final best:", values[:, -1].tolist())

## Plot mean and standard error

The metric is **noise-free best-so-far Strength**, including the initial design. Standard error is not a confidence interval. Curves may cross; no method is assumed to win.

In [ ]:
iterations = list(range(STEPS + 1))
fig, ax = plt.subplots(figsize=(8, 4.5))
for name, values in curves.items():
    mean, std, sem = summarize_trajectories(values)
    center = mean.cpu().numpy()
    spread = sem.cpu().numpy()
    line, = ax.plot(iterations, center, marker="o", label=name)
    ax.fill_between(iterations, center - spread, center + spread, alpha=0.15, color=line.get_color())
ax.set(xlabel="Additional evaluations", ylabel="Best-so-far true Strength")
ax.legend()
ax.grid(alpha=0.25)
plt.show()

## Paired differences at the final budget

Paired differences use the same seed and initial design. Inspect their distribution instead of treating overlapping SEM bands as a significance test.

In [ ]:
difference = curves["qei"][:, -1] - curves["random"][:, -1]
print("qEI - Random final paired differences:", difference.tolist())
print("Mean paired difference:", difference.mean().item())
print("Sample standard deviation:", difference.std(unbiased=True).item())

## Interpretation and limitations

The default three seeds are illustrative, not sufficient to claim superiority. The benchmark is a noise-free oracle evaluation metric; the acquisition sees only noisy observations. Computational runtime, constraint feasibility, and multiobjective hypervolume are **not** measured. See `docs/optimization/heterogeneous-e2e-benchmark-report.md` for a reporting template and the remaining E2E coverage.